# Universal STFT quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mananmittal/Universal-Short-Time-Fourier-Transform/blob/main/examples/quickstart.ipynb)

The **Universal Short-Time Fourier Transform (U-STFT)** is an STFT that picks
its window length as it goes: long windows during steady sounds (sharp in
frequency), short windows at clicks and onsets (sharp in time).

This notebook takes about a minute to run. It:

1. builds a test signal with two close tones followed by a train of clicks;
2. compares the U-STFT with a short and a long fixed-window STFT;
3. shows which window length the U-STFT chose over time;
4. filters the signal and turns it back into audio exactly;
5. runs the same thing with the PyTorch backend.

Code, paper and results: [GitHub](https://github.com/mananmittal/Universal-Short-Time-Fourier-Transform) ·
[project page](https://mananmittal.github.io/Universal-Short-Time-Fourier-Transform/).

## 1. Install

On Colab this installs the package straight from GitHub. If `universal_stft`
is already installed (for example you cloned the repository and ran
`pip install -e .`), the cell does nothing.

In [ ]:
try:
    import universal_stft
except ImportError:
    %pip install -q git+https://github.com/mananmittal/Universal-Short-Time-Fourier-Transform.git
    import universal_stft

print("universal_stft", universal_stft.__version__)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft as fixed_stft

from universal_stft import universal_short_time_fourier_transform, inverse_universal_spectrogram

## 2. A test signal

One second of two tones only 150 Hz apart (1000 Hz and 1150 Hz), then one
second of short clicks, every 0.1 s. A long window is needed to separate the
tones, and a short window is needed to keep each click sharp in time. No single
fixed window does both.

In [ ]:
fs = 8000                                   # sample rate (Hz)
t = np.arange(2 * fs) / fs                  # 2 seconds

tones = np.sin(2 * np.pi * 1000 * t) + np.sin(2 * np.pi * 1150 * t)
x = np.where(t < 1.0, tones, 0.0)           # tones for the first second
x[fs + 400::800] += 2.0                     # then a click every 0.1 s
x += 1e-3 * np.random.default_rng(0).standard_normal(len(x))   # a little noise

plt.figure(figsize=(9, 2))
plt.plot(t, x, lw=0.5)
plt.xlabel("Time (s)")
plt.title("Test signal: two close tones, then clicks")
plt.show()

## 3. Run the U-STFT (analysis mode)

`hop_size` is the step between frames **and** the shortest window. `n_fft`
sets the longest window: the bank is `hop_size, 2*hop_size, 4*hop_size, ...`
up to `n_fft // 2`. With `n_fft=512, hop_size=32` the candidates are 32, 64,
128 and 256 samples. The default `mode="analysis"` makes one causal pass.

In [ ]:
n_fft, hop_size = 512, 32
result = universal_short_time_fourier_transform(x, n_fft=n_fft, hop_size=hop_size)

S = result.analysis_spectrogram             # complex, shape (frames, bins)
print("spectrogram shape:", S.shape)
print("window bank (samples):", result.blend_info[0][0])

## 4. Compare with fixed-window STFTs

Below, the U-STFT sits next to an ordinary STFT with a short window (32
samples, 4 ms) and one with a long window (256 samples, 32 ms), both from
`scipy.signal.stft` with the same hop. Look at two things: are the two tones
separated in the first second, and are the clicks thin vertical lines in the
second? The plot is zoomed in on the moment the tones stop and the clicks
start. Every frame is placed at the time of its newest sample, so a click shows
up at its arrival time and lasts as long as the window that contains it.

In [ ]:
def to_db(Z):
    Z_db = 20 * np.log10(np.abs(Z) + 1e-12)
    return Z_db - Z_db.max()

frame_times = (np.arange(S.shape[0]) + 1) * hop_size / fs   # end time of each frame (s)
freqs = np.arange(S.shape[1]) * fs / n_fft

panels = [("U-STFT (adaptive)", frame_times, freqs, to_db(S.T))]
for L in (32, 256):
    f, tt, Z = fixed_stft(x, fs=fs, window=("gaussian", L / 6), nperseg=L,
                          noverlap=L - hop_size, nfft=n_fft, boundary=None, padded=False)
    tt_end = tt + (L / 2) / fs              # scipy times frames by their centre; use the newest sample, like the U-STFT
    panels.append((f"Fixed STFT, window {L} samples", tt_end, f, to_db(Z)))

# Zoom in on the hand-over from tones to clicks: 0.8 s to 1.3 s, 500 Hz to 1700 Hz.
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True, constrained_layout=True)
for ax, (title, tt, ff, Z_db) in zip(axes, panels):
    im = ax.pcolormesh(tt, ff, Z_db, vmin=-50, vmax=0, cmap="magma", shading="auto")
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_xlim(0.8, 1.3)
    ax.set_ylim(500, 1700)
axes[0].set_ylabel("Frequency (Hz)")
fig.colorbar(im, ax=axes, label="dB (relative to peak)")
plt.show()

The short window keeps the clicks sharp but merges the two tones into one
blurry band. The long window separates the tones (the dark line between them)
but smears every click over 32 ms. The U-STFT uses the long window on the tones,
so they stay separated, and keeps switching back to short windows in the click
section, so the clicks come out narrower than with the long fixed window. They
are not as thin as with the 32-sample window: the analysis is causal, so a
click has to arrive before the window can shrink.

## 5. Which window did it choose?

`dominant_lengths` is the most likely window length at each frame and
`pmf_history` is the full probability over the window bank (each row sums to 1).

In [ ]:
lengths = result.blend_info[0][0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax1.step(frame_times, result.dominant_lengths, where="post")
ax1.set_yscale("log", base=2)
ax1.set_yticks(lengths, [str(L) for L in lengths])
ax1.set_ylabel("Most likely\nwindow (samples)")

ax2.imshow(result.pmf_history.T, origin="lower", aspect="auto", cmap="Greys",
           interpolation="nearest",
           extent=[frame_times[0], frame_times[-1], -0.5, len(lengths) - 0.5])
ax2.set_yticks(range(len(lengths)), [str(L) for L in lengths])
ax2.set_ylabel("Probability\nper window")
ax2.set_xlabel("Time (s)")
plt.show()

During the tones the window climbs to 256 samples within a few frames and
stays there. In the click section it keeps dropping back to the short windows.

## 6. Filter and resynthesize (filtering mode)

The adaptive spectrogram above is for looking at, not for inverting: its window
changes from frame to frame. Run with `mode="filtering"` to also get a
companion stream, `filtering_spectrogram`, that inverts exactly. Always pass the
same `n_fft` and `hop_size` to the inverse.

In [ ]:
result = universal_short_time_fourier_transform(x, n_fft=n_fft, hop_size=hop_size, mode="filtering")
F = result.filtering_spectrogram            # same shape as the analysis spectrogram

x_hat = inverse_universal_spectrogram(F, len(x), n_fft=n_fft, hop_size=hop_size)
print("max reconstruction error:", np.max(np.abs(x - x_hat)))

To filter, multiply **every frame by the same gain per frequency bin**, then
invert. Here is a low-pass filter at 1075 Hz, which keeps the 1000 Hz tone, removes
the 1150 Hz tone and strips the high-frequency energy of the clicks.

In [ ]:
gain = (freqs < 1075).astype(float)         # one gain per frequency bin
x_low = inverse_universal_spectrogram(F * gain, len(x), n_fft=n_fft, hop_size=hop_size)

# Spectrum of a steady stretch of the tones (0.25 s to 0.75 s), with a Hann window.
seg = slice(fs // 4, 3 * fs // 4)
taper = np.hanning(seg.stop - seg.start)
f = np.fft.rfftfreq(seg.stop - seg.start, 1 / fs)
spectrum_db = lambda y: 20 * np.log10(np.abs(np.fft.rfft(y[seg] * taper)) + 1e-12)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5), constrained_layout=True)
ax1.plot(f, spectrum_db(x), label="original", lw=0.8)
ax1.plot(f, spectrum_db(x_low), label="low-passed", lw=0.8)
ax1.set_xlim(800, 1400)
ax1.set_ylim(-120, 70)
ax1.set_xlabel("Frequency (Hz)"); ax1.set_ylabel("dB"); ax1.legend()
ax1.set_title("Spectrum of the tones")
ax2.plot(t, x, lw=0.8, label="original")
ax2.plot(t, x_low, lw=0.8, label="low-passed")
ax2.set_xlim(1.045, 1.055)
ax2.set_xlabel("Time (s)"); ax2.legend()
ax2.set_title("One click, zoomed in")
plt.show()

The 1000 Hz tone passes unchanged and the 1150 Hz tone drops by about 50 dB.
The click keeps its timing but loses its high frequencies, so it becomes a
smaller, smoother pulse.

Optional: listen to the original and the filtered signal (works in Jupyter and Colab).

In [ ]:
from IPython.display import Audio, display
display(Audio(x, rate=fs), Audio(x_low, rate=fs))

## 7. PyTorch backend (optional)

`universal_stft.torch` has the same function names, arguments and output
fields. It takes `(samples,)` or `(batch, samples)` tensors, keeps the device
and precision, and is differentiable. Colab ships with PyTorch; elsewhere this
cell is skipped if PyTorch is not installed.

In [ ]:
try:
    import torch
except ImportError:
    print("PyTorch is not installed; skipping. Install it with: pip install torch")
else:
    from universal_stft.torch import universal_short_time_fourier_transform as ustft_torch

    xt = torch.tensor(np.stack([x, x[::-1].copy()]), dtype=torch.float32)   # a batch of 2 signals
    res = ustft_torch(xt, n_fft=n_fft, hop_size=hop_size)
    print(res.analysis_spectrogram.shape, res.analysis_spectrogram.dtype)

## Next steps

* The [README](https://github.com/mananmittal/Universal-Short-Time-Fourier-Transform#readme)
  has a step-by-step guide, how to choose `n_fft`, `hop_size` and `beta`,
  multichannel use, the full API and a troubleshooting FAQ.
* The [project page](https://mananmittal.github.io/Universal-Short-Time-Fourier-Transform/)
  shows results on whale song, snapping shrimp, fish choruses and speech.
* If the U-STFT is useful in your work, please cite the paper
  (see the [Citation section](https://github.com/mananmittal/Universal-Short-Time-Fourier-Transform#citation)).